### HITL(Human-in-the-loop)

검토가 필요할 수 있는 작업(예: 파일 쓰기 또는 SQL 실행)의 경우, 일시 중지(인터럽트)하고 사람이 개입할 수 있게 합니다.


> https://docs.langchain.com/oss/python/langchain/human-in-the-loop

In [3]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [4]:
from langchain_core.tools import tool

@tool
def write_file_tool(filename: str, content: str) -> str:
    """파일을 지정된 경로에 작성합니다. (이름: write_file)"""
    return f"파일 '{filename}'에 내용이 성공적으로 기록되었습니다."

@tool
def execute_sql_tool(query: str) -> str:
    """데이터베이스에서 SQL 쿼리를 실행합니다. (이름: execute_sql)"""
    return f"쿼리 '{query}'가 실행되었습니다. (영향을 받은 행: 1개)"

@tool
def read_data_tool(source: str) -> str:
    """지정된 소스에서 데이터를 읽어옵니다. (이름: read_data)"""
    return f"'{source}'로부터 데이터를 성공적으로 불러왔습니다: [샘플 데이터]"

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 


agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[write_file_tool, execute_sql_tool, read_data_tool],
    middleware=[
        HumanInTheLoopMiddleware( 
            interrupt_on={
                "write_file_tool": True,  # 승인, 수정, 거절(approve, edit, reject) 가능한 설정
                "execute_sql_tool": {"allowed_decisions": ["approve", "reject"]},  # 승인, 거절(approve, reject) 가능한 설정
                "read_data_tool": False, # 사용자 승인 없이 즉시 실행
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
    system_prompt="모든 답변은 한국어로 작성해주세요." 
)

In [6]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "abc 테이블의 모든 데이터를 삭제해줘",
            }
        ]
    },
    config={"configurable": {"thread_id": "1"}}  
)


In [7]:
result

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘', additional_kwargs={}, response_metadata={}, id='ae53ec83-7344-40e4-ab56-443e1b98fad5'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc"}'}, '__gemini_function_call_thought_signatures__': {'290790e8-2a4f-47ae-9289-be68c8ff3ca8': 'CrUCAb4+9vvmWQyJLnjmTc3RHyfSmZRPbSm0AeAPzcwdZD11L6D/jBbGPqmeoBr8ulIt4ONIPQdzk2AQ1c3qSjaGjbAtwoSf/SGMxAiEit648iQ6eQrRqjg/pNBlEkNEzTOpnXWOeioPLIARfjUE3FxTZErmX8X9hPp3tdsqZkdF+AjMK+hhBFQWTGB0sClnGF5oldUeq5ETNN1EofYa758fyY7vPEzhkm80KmERipIqhhK+dA44o6QCmFb859HQ5Z0L8iYGPKt+iEp6XsIiP8Jh+fJRHtKrDhh7ZA9vSE8A3AW3YcFGTmsl6B6fdrRMwn0v5kv1syqsG2D1t7WXOOKzO8MV5RqMW3wrdCk0rfQI8Ge29kzr/yfFr1SHrVV0Co8lEhAu/kKv203RADgpxw83QvzV6gXL'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cfc2a-a7af-7f10-b002-eb049a516a06-0', tool_calls=[{'name'

In [8]:
result['__interrupt__']

[Interrupt(value={'action_requests': [{'name': 'execute_sql_tool', 'args': {'query': 'DELETE FROM abc'}, 'description': "Tool execution pending approval\n\nTool: execute_sql_tool\nArgs: {'query': 'DELETE FROM abc'}"}], 'review_configs': [{'action_name': 'execute_sql_tool', 'allowed_decisions': ['approve', 'reject']}]}, id='b011f8f2b3ab29c121a024968cb29300')]

In [ ]:
from langgraph.types import Command

agent.invoke(
    Command( 
        resume={"decisions": [{"type": "reject"}]}
    ), 
    config={"configurable": {"thread_id": "1"}} 
)

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘', additional_kwargs={}, response_metadata={}, id='ae53ec83-7344-40e4-ab56-443e1b98fad5'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc"}'}, '__gemini_function_call_thought_signatures__': {'290790e8-2a4f-47ae-9289-be68c8ff3ca8': 'CrUCAb4+9vvmWQyJLnjmTc3RHyfSmZRPbSm0AeAPzcwdZD11L6D/jBbGPqmeoBr8ulIt4ONIPQdzk2AQ1c3qSjaGjbAtwoSf/SGMxAiEit648iQ6eQrRqjg/pNBlEkNEzTOpnXWOeioPLIARfjUE3FxTZErmX8X9hPp3tdsqZkdF+AjMK+hhBFQWTGB0sClnGF5oldUeq5ETNN1EofYa758fyY7vPEzhkm80KmERipIqhhK+dA44o6QCmFb859HQ5Z0L8iYGPKt+iEp6XsIiP8Jh+fJRHtKrDhh7ZA9vSE8A3AW3YcFGTmsl6B6fdrRMwn0v5kv1syqsG2D1t7WXOOKzO8MV5RqMW3wrdCk0rfQI8Ge29kzr/yfFr1SHrVV0Co8lEhAu/kKv203RADgpxw83QvzV6gXL'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cfc2a-a7af-7f10-b002-eb049a516a06-0', tool_calls=[{'name'

In [10]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "abc 테이블의 모든 데이터를 삭제해줘",
            }
        ]
    },
    config={"configurable": {"thread_id": "1"}}  
)


In [11]:
from langgraph.types import Command

agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config={"configurable": {"thread_id": "1"}} 
)

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘', additional_kwargs={}, response_metadata={}, id='ae53ec83-7344-40e4-ab56-443e1b98fad5'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc"}'}, '__gemini_function_call_thought_signatures__': {'290790e8-2a4f-47ae-9289-be68c8ff3ca8': 'CrUCAb4+9vvmWQyJLnjmTc3RHyfSmZRPbSm0AeAPzcwdZD11L6D/jBbGPqmeoBr8ulIt4ONIPQdzk2AQ1c3qSjaGjbAtwoSf/SGMxAiEit648iQ6eQrRqjg/pNBlEkNEzTOpnXWOeioPLIARfjUE3FxTZErmX8X9hPp3tdsqZkdF+AjMK+hhBFQWTGB0sClnGF5oldUeq5ETNN1EofYa758fyY7vPEzhkm80KmERipIqhhK+dA44o6QCmFb859HQ5Z0L8iYGPKt+iEp6XsIiP8Jh+fJRHtKrDhh7ZA9vSE8A3AW3YcFGTmsl6B6fdrRMwn0v5kv1syqsG2D1t7WXOOKzO8MV5RqMW3wrdCk0rfQI8Ge29kzr/yfFr1SHrVV0Co8lEhAu/kKv203RADgpxw83QvzV6gXL'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cfc2a-a7af-7f10-b002-eb049a516a06-0', tool_calls=[{'name'